# 17 - Evolução das Previsões Rodada a Rodada

Simulação Monte Carlo aplicada **a cada rodada** do Brasileirão 2026.
Mostra como as probabilidades de título, Libertadores e rebaixamento evoluem
à medida que os jogos acontecem.

- Evolução das chances de campeão
- Evolução das chances de rebaixamento
- Previsão de colocação final por rodada
- Heatmap de posições previstas por rodada

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

np.random.seed(42)

df = pd.read_csv('../data/raw/campeonato-brasileiro-full.csv')
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
df['ano'] = df['data'].dt.year
df = df.dropna(subset=['rodata'])
df['rodata'] = df['rodata'].astype(int)

ANO_ATUAL = 2026
N_SIMULACOES = 5000  # menos que o nb15 para rodar mais rápido (por rodada)
TOTAL_RODADAS = 38

df_historico = df[df['ano'] < ANO_ATUAL].copy()
df_atual = df[df['ano'] == ANO_ATUAL].copy()

rodada_max = int(df_atual['rodata'].max())
times_2026 = sorted(set(df_atual['mandante'].unique()) | set(df_atual['visitante'].unique()))

print(f'Rodadas disponíveis: 1 a {rodada_max}')
print(f'Times: {len(times_2026)}')
print(f'Simulações por rodada: {N_SIMULACOES:,}')

Rodadas disponíveis: 1 a 6
Times: 20
Simulações por rodada: 5,000


In [2]:
# === FUNÇÕES BASE (do notebook 15) ===

media_gols_liga = df_historico['mandante_Placar'].mean()
media_gols_visitante_liga = df_historico['visitante_Placar'].mean()


def calcular_forca_historica(df_hist, times, anos_recentes=4):
    df_rec = df_hist[df_hist['ano'] >= df_hist['ano'].max() - anos_recentes + 1]
    forca = {}
    for time in times:
        mand = df_rec[df_rec['mandante'] == time]
        visit = df_rec[df_rec['visitante'] == time]
        jogos = len(mand) + len(visit)
        if jogos == 0:
            forca[time] = {'ataque_m': 1.3, 'defesa_m': 1.1,
                           'ataque_v': 1.0, 'defesa_v': 1.3}
            continue
        forca[time] = {
            'ataque_m': mand['mandante_Placar'].mean() if len(mand) > 0 else 1.3,
            'defesa_m': mand['visitante_Placar'].mean() if len(mand) > 0 else 1.1,
            'ataque_v': visit['visitante_Placar'].mean() if len(visit) > 0 else 1.0,
            'defesa_v': visit['mandante_Placar'].mean() if len(visit) > 0 else 1.3,
        }
    return forca


def calcular_classificacao(df_jogos):
    times = set(df_jogos['mandante'].unique()) | set(df_jogos['visitante'].unique())
    stats = {t: {'pts': 0, 'v': 0, 'sg': 0, 'gp': 0, 'j': 0} for t in times}
    for _, jogo in df_jogos.iterrows():
        m, vis = jogo['mandante'], jogo['visitante']
        gm, gv = int(jogo['mandante_Placar']), int(jogo['visitante_Placar'])
        stats[m]['j'] += 1; stats[vis]['j'] += 1
        stats[m]['gp'] += gm; stats[vis]['gp'] += gv
        stats[m]['sg'] += gm - gv; stats[vis]['sg'] += gv - gm
        if gm > gv: stats[m]['pts'] += 3; stats[m]['v'] += 1
        elif gv > gm: stats[vis]['pts'] += 3; stats[vis]['v'] += 1
        else: stats[m]['pts'] += 1; stats[vis]['pts'] += 1
    return stats


def simular_jogo(mandante, visitante, forca):
    fm = forca.get(mandante, {})
    fv = forca.get(visitante, {})
    ataque_m = fm.get('ataque_m', 1.3)
    defesa_v = fv.get('defesa_v', 1.3)
    lambda_m = (ataque_m / media_gols_liga) * (defesa_v / media_gols_visitante_liga) * media_gols_liga
    ataque_v = fv.get('ataque_v', 1.0)
    defesa_m = fm.get('defesa_m', 1.1)
    lambda_v = (ataque_v / media_gols_visitante_liga) * (defesa_m / media_gols_liga) * media_gols_visitante_liga
    lambda_m = np.clip(lambda_m, 0.3, 4.0)
    lambda_v = np.clip(lambda_v, 0.2, 3.5)
    return np.random.poisson(lambda_m), np.random.poisson(lambda_v)


def simular_campeonato(stats_atuais, jogos_restantes, forca, times):
    stats = {t: dict(s) for t, s in stats_atuais.items()}
    for mandante, visitante in jogos_restantes:
        gm, gv = simular_jogo(mandante, visitante, forca)
        stats[mandante]['gp'] += gm; stats[visitante]['gp'] += gv
        stats[mandante]['sg'] += gm - gv; stats[visitante]['sg'] += gv - gm
        if gm > gv: stats[mandante]['pts'] += 3; stats[mandante]['v'] += 1
        elif gv > gm: stats[visitante]['pts'] += 3; stats[visitante]['v'] += 1
        else: stats[mandante]['pts'] += 1; stats[visitante]['pts'] += 1
    ranking = sorted(stats.items(),
                     key=lambda x: (x[1]['pts'], x[1]['v'], x[1]['sg'], x[1]['gp']),
                     reverse=True)
    return {time: pos+1 for pos, (time, _) in enumerate(ranking)}, \
           {time: s['pts'] for time, s in stats.items()}


forca = calcular_forca_historica(df_historico, times_2026)
print('Força dos times calculada.')

Força dos times calculada.


In [3]:
# === SIMULAR PARA CADA RODADA ===
# Para cada rodada jogada, simular o restante do campeonato

# Todos os jogos do campeonato (ida e volta)
todos_jogos = set()
for _, j in df_atual.iterrows():
    todos_jogos.add((j['mandante'], j['visitante']))

# Adicionar jogos que ainda não aconteceram
for m in times_2026:
    for v in times_2026:
        if m != v:
            todos_jogos.add((m, v))

# Resultados por rodada
evolucao = {t: {'titulo': [], 'g6': [], 'z4': [], 'pos_media': [], 'pts_media': []} for t in times_2026}
rodadas_simuladas = []

for rodada in range(1, rodada_max + 1):
    print(f'Simulando a partir da rodada {rodada}...', end=' ')
    
    # Jogos até esta rodada
    df_ate_rodada = df_atual[df_atual['rodata'] <= rodada]
    stats = calcular_classificacao(df_ate_rodada)
    
    # Garantir todos os times
    for t in times_2026:
        if t not in stats:
            stats[t] = {'pts': 0, 'v': 0, 'sg': 0, 'gp': 0, 'j': 0}
    
    # Jogos já feitos até esta rodada
    jogos_feitos = set()
    for _, j in df_ate_rodada.iterrows():
        jogos_feitos.add((j['mandante'], j['visitante']))
    
    # Jogos restantes
    jogos_rest = [(m, v) for m, v in todos_jogos if (m, v) not in jogos_feitos]
    
    # Simular
    posicoes = {t: [] for t in times_2026}
    pontos = {t: [] for t in times_2026}
    
    for _ in range(N_SIMULACOES):
        ranking, pts = simular_campeonato(stats, jogos_rest, forca, times_2026)
        for t in times_2026:
            posicoes[t].append(ranking[t])
            pontos[t].append(pts[t])
    
    # Registrar
    rodadas_simuladas.append(rodada)
    for t in times_2026:
        pos_arr = np.array(posicoes[t])
        pts_arr = np.array(pontos[t])
        evolucao[t]['titulo'].append((pos_arr == 1).mean() * 100)
        evolucao[t]['g6'].append((pos_arr <= 6).mean() * 100)
        evolucao[t]['z4'].append((pos_arr >= 17).mean() * 100)
        evolucao[t]['pos_media'].append(pos_arr.mean())
        evolucao[t]['pts_media'].append(pts_arr.mean())
    
    print('OK')

print(f'\nSimulação concluída para {len(rodadas_simuladas)} rodadas.')

Simulando a partir da rodada 1... 

OK
Simulando a partir da rodada 2... 

OK
Simulando a partir da rodada 3... 

OK
Simulando a partir da rodada 4... 

OK
Simulando a partir da rodada 5... 

OK
Simulando a partir da rodada 6... 

OK

Simulação concluída para 6 rodadas.


In [4]:
# === GRÁFICO 1: EVOLUÇÃO DA CHANCE DE TÍTULO ===

# Top times com chance de título
top_titulo = sorted(times_2026, key=lambda t: evolucao[t]['titulo'][-1], reverse=True)[:8]
top_titulo = [t for t in top_titulo if evolucao[t]['titulo'][-1] > 0.5]

cores_times = {
    'Palmeiras': '#006437', 'Flamengo': '#8C1616', 'São Paulo': '#FF0000',
    'Corinthians': '#1C1C1C', 'Botafogo': '#4A4A4A', 'Fluminense': '#7B2D4E',
    'Atlético-MG': '#2E2E2E', 'Internacional': '#E30613', 'Grêmio': '#0A6AB6',
    'Cruzeiro': '#003DA5', 'Santos': '#6B6B6B', 'Vasco': '#333333',
    'Bahia': '#004A99', 'Athletico-PR': '#AF1E2D', 'Mirassol': '#FFD700',
    'Coritiba': '#006631', 'Bragantino': '#B22222', 'Vitória': '#FF4500',
    'Chapecoense': '#006633', 'Remo': '#003DA5',
}

# Padrões de linha para times com cores similares
dash_times = {
    'Corinthians': 'solid', 'Botafogo': 'dash', 'Santos': 'dot', 'Vasco': 'dashdot',
    'Atlético-MG': 'longdash',
}

fig_tit = go.Figure()
for time in top_titulo:
    fig_tit.add_trace(go.Scatter(
        x=rodadas_simuladas,
        y=evolucao[time]['titulo'],
        mode='lines+markers',
        name=time,
        line=dict(color=cores_times.get(time, '#888'), width=2.5,
                  dash=dash_times.get(time, 'solid')),
        marker=dict(size=8),
        hovertemplate=f'{time}<br>Rodada %{{x}}<br>Chance título: %{{y:.1f}}%<extra></extra>'
    ))

fig_tit.update_layout(
    title=f'Evolução da Chance de Título — Brasileirão 2026<br>'
          f'<sup>Monte Carlo ({N_SIMULACOES:,} simulações por rodada) | Rodadas 1-{rodada_max}</sup>',
    xaxis_title='Rodada',
    yaxis_title='Probabilidade de Título (%)',
    template='plotly_white',
    width=1000, height=600,
    xaxis=dict(dtick=1),
    hovermode='x unified',
    legend=dict(x=1.02, y=1)
)

fig_tit.show()
path = '../charts/evolucao_titulo.html'
fig_tit.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Série temporal da probabilidade de título via simulação Monte Carlo, evidenciando a convergência ou divergência das chances ao longo das rodadas."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/evolucao_titulo.html')

Gráfico exportado: charts/evolucao_titulo.html


In [5]:
# === GRÁFICO 2: EVOLUÇÃO DO RISCO DE REBAIXAMENTO ===

top_z4 = sorted(times_2026, key=lambda t: evolucao[t]['z4'][-1], reverse=True)[:8]
top_z4 = [t for t in top_z4 if evolucao[t]['z4'][-1] > 5]

fig_z4 = go.Figure()
for time in top_z4:
    fig_z4.add_trace(go.Scatter(
        x=rodadas_simuladas,
        y=evolucao[time]['z4'],
        mode='lines+markers',
        name=time,
        line=dict(color=cores_times.get(time, '#888'), width=2.5,
                  dash=dash_times.get(time, 'solid')),
        marker=dict(size=8),
        hovertemplate=f'{time}<br>Rodada %{{x}}<br>Risco Z4: %{{y:.1f}}%<extra></extra>'
    ))

# Linha de referência 50%
fig_z4.add_hline(y=50, line_dash='dot', line_color='red', opacity=0.4,
                 annotation_text='50%')

fig_z4.update_layout(
    title=f'Evolução do Risco de Rebaixamento (Z4) — Brasileirão 2026<br>'
          f'<sup>Monte Carlo ({N_SIMULACOES:,} simulações por rodada) | Rodadas 1-{rodada_max}</sup>',
    xaxis_title='Rodada',
    yaxis_title='Probabilidade de Z4 (%)',
    template='plotly_white',
    width=1000, height=600,
    xaxis=dict(dtick=1),
    hovermode='x unified',
    legend=dict(x=1.02, y=1)
)

fig_z4.show()
path = '../charts/evolucao_z4.html'
fig_z4.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Evolução da probabilidade de rebaixamento (Z4) por rodada via Monte Carlo, com linha de referência em 50% para identificar times em zona crítica."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/evolucao_z4.html')

Gráfico exportado: charts/evolucao_z4.html


In [6]:
# === GRÁFICO 3: EVOLUÇÃO DA POSIÇÃO PREVISTA ===
# Bump chart da posição média prevista por rodada (Top 6 + Bottom 4)

destaque_pos = sorted(times_2026, key=lambda t: evolucao[t]['pos_media'][-1])
times_bump = destaque_pos[:6] + destaque_pos[-4:]
times_fundo = [t for t in times_2026 if t not in times_bump]

fig_bump = go.Figure()

# Times de fundo (opacidade reduzida, sem legenda)
for time in times_fundo:
    fig_bump.add_trace(go.Scatter(
        x=rodadas_simuladas,
        y=evolucao[time]['pos_media'],
        mode='lines',
        name=time,
        line=dict(color='#CCCCCC', width=1),
        opacity=0.4,
        showlegend=False,
        hovertemplate=f'{time}<br>Rodada %{{x}}<br>Posição prevista: %{{y:.1f}}°<extra></extra>'
    ))

# Times em destaque
for time in times_bump:
    fig_bump.add_trace(go.Scatter(
        x=rodadas_simuladas,
        y=evolucao[time]['pos_media'],
        mode='lines+markers',
        name=time,
        line=dict(color=cores_times.get(time, '#888'), width=2.5,
                  dash=dash_times.get(time, 'solid')),
        marker=dict(size=6),
        hovertemplate=f'{time}<br>Rodada %{{x}}<br>Posição prevista: %{{y:.1f}}°<extra></extra>'
    ))

# Zonas
fig_bump.add_hrect(y0=0.5, y1=6.5, fillcolor='green', opacity=0.05,
                   annotation_text='G6', annotation_position='top left')
fig_bump.add_hrect(y0=16.5, y1=20.5, fillcolor='red', opacity=0.05,
                   annotation_text='Z4', annotation_position='bottom left')

fig_bump.update_layout(
    title=f'Evolução da Posição Prevista — Brasileirão 2026<br>'
          f'<sup>Top 6 e Bottom 4 em destaque | Monte Carlo | Rodadas 1-{rodada_max}</sup>',
    xaxis_title='Rodada',
    yaxis_title='Posição Prevista (média)',
    yaxis=dict(autorange='reversed', dtick=1),
    template='plotly_white',
    width=1100, height=800,
    xaxis=dict(dtick=1),
    hovermode='x unified',
    legend=dict(x=1.02, y=1)
)

fig_bump.show()
path = '../charts/evolucao_posicao.html'
fig_bump.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Bump chart da posição média prevista por simulação Monte Carlo, mostrando a evolução da classificação esperada rodada a rodada."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/evolucao_posicao.html')

Gráfico exportado: charts/evolucao_posicao.html


In [7]:
# === GRÁFICO 4: HEATMAP — PROBABILIDADE DE G6 POR RODADA ===

# Ordenar times pela posição prevista na última rodada
ordem = sorted(times_2026, key=lambda t: evolucao[t]['pos_media'][-1])

# Matriz: times x rodadas → prob G6
matriz_g6 = np.zeros((len(ordem), len(rodadas_simuladas)))
for i, time in enumerate(ordem):
    for j, _ in enumerate(rodadas_simuladas):
        matriz_g6[i, j] = evolucao[time]['g6'][j]

fig_hg6 = go.Figure(data=go.Heatmap(
    z=matriz_g6,
    x=[f'R{r}' for r in rodadas_simuladas],
    y=ordem,
    colorscale=[
        [0, '#1a1a2e'],
        [0.3, '#16213e'],
        [0.5, '#0f3460'],
        [0.7, '#238636'],
        [0.9, '#3fb950'],
        [1.0, '#7ee787']
    ],
    text=np.round(matriz_g6, 1),
    texttemplate='%{text}%',
    textfont={'size': 9},
    hovertemplate='%{y}<br>Rodada %{x}<br>Prob. G6: %{z:.1f}%<extra></extra>',
    colorbar=dict(title='Prob G6 (%)')
))

fig_hg6.update_layout(
    title=f'Evolução da Probabilidade de Libertadores (G6) — Brasileirão 2026<br>'
          f'<sup>{N_SIMULACOES:,} simulações por rodada</sup>',
    xaxis_title='Rodada',
    template='plotly_white',
    width=900, height=700,
)

fig_hg6.show()
path = '../charts/evolucao_g6_heatmap.html'
fig_hg6.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Heatmap da probabilidade de Libertadores (G6) por time e rodada, baseado em simulações Monte Carlo — intensidade indica convergência das projeções."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/evolucao_g6_heatmap.html')

Gráfico exportado: charts/evolucao_g6_heatmap.html


In [8]:
# === GRÁFICO 5: HEATMAP — RISCO DE REBAIXAMENTO POR RODADA ===

# Só times com risco significativo
times_risco = [t for t in ordem if evolucao[t]['z4'][-1] > 3]
# Ordenar por risco crescente
times_risco = sorted(times_risco, key=lambda t: evolucao[t]['z4'][-1])

matriz_z4 = np.zeros((len(times_risco), len(rodadas_simuladas)))
for i, time in enumerate(times_risco):
    for j, _ in enumerate(rodadas_simuladas):
        matriz_z4[i, j] = evolucao[time]['z4'][j]

fig_hz4 = go.Figure(data=go.Heatmap(
    z=matriz_z4,
    x=[f'R{r}' for r in rodadas_simuladas],
    y=times_risco,
    colorscale=[
        [0, '#0d1117'],
        [0.2, '#3d1a1a'],
        [0.4, '#8b2020'],
        [0.6, '#da3633'],
        [0.8, '#f85149'],
        [1.0, '#ff7b72']
    ],
    text=np.round(matriz_z4, 1),
    texttemplate='%{text}%',
    textfont={'size': 10},
    hovertemplate='%{y}<br>Rodada %{x}<br>Risco Z4: %{z:.1f}%<extra></extra>',
    colorbar=dict(title='Risco Z4 (%)')
))

fig_hz4.update_layout(
    title=f'Evolução do Risco de Rebaixamento — Brasileirão 2026<br>'
          f'<sup>{N_SIMULACOES:,} simulações por rodada</sup>',
    xaxis_title='Rodada',
    template='plotly_white',
    width=900, height=600,
)

fig_hz4.show()
path = '../charts/evolucao_z4_heatmap.html'
fig_hz4.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Heatmap do risco de rebaixamento por time e rodada — escala de cor indica a intensidade da probabilidade estimada por simulação."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/evolucao_z4_heatmap.html')

Gráfico exportado: charts/evolucao_z4_heatmap.html


In [9]:
# === GRÁFICO 6: EVOLUÇÃO DOS PONTOS PREVISTOS ===

fig_pts = go.Figure()

# Top 6 e bottom 4
destaque = sorted(times_2026, key=lambda t: evolucao[t]['pos_media'][-1])
times_destaque = destaque[:6] + destaque[-4:]

for time in times_destaque:
    fig_pts.add_trace(go.Scatter(
        x=rodadas_simuladas,
        y=evolucao[time]['pts_media'],
        mode='lines+markers',
        name=time,
        line=dict(color=cores_times.get(time, '#888'), width=2.5,
                  dash=dash_times.get(time, 'solid')),
        marker=dict(size=7),
        hovertemplate=f'{time}<br>Rodada %{{x}}<br>Pts previstos: %{{y:.1f}}<extra></extra>'
    ))

# Linhas de referência
fig_pts.add_hline(y=45, line_dash='dot', line_color='red', opacity=0.3,
                  annotation_text='~Rebaixamento (45 pts)')
fig_pts.add_hline(y=60, line_dash='dot', line_color='green', opacity=0.3,
                  annotation_text='~Libertadores (60 pts)')

fig_pts.update_layout(
    title=f'Evolução da Pontuação Prevista — Brasileirão 2026<br>'
          f'<sup>Top 6 e Bottom 4 | Monte Carlo por rodada</sup>',
    xaxis_title='Rodada',
    yaxis_title='Pontos Previstos (média)',
    template='plotly_white',
    width=1000, height=600,
    xaxis=dict(dtick=1),
    hovermode='x unified',
    legend=dict(x=1.02, y=1)
)

fig_pts.show()
path = '../charts/evolucao_pontos.html'
fig_pts.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Série temporal da pontuação média prevista via Monte Carlo para os times em destaque, com limiares de referência para Libertadores (~60 pts) e rebaixamento (~45 pts)."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/evolucao_pontos.html')

Gráfico exportado: charts/evolucao_pontos.html


In [10]:
# === RESUMO FINAL ===

print('=' * 60)
print(f'EVOLUÇÃO DAS PREVISÕES — Brasileirão 2026')
print(f'Rodadas analisadas: 1 a {rodada_max}')
print('=' * 60)

print(f'\nChance de TÍTULO após rodada {rodada_max}:')
for t in sorted(times_2026, key=lambda t: evolucao[t]['titulo'][-1], reverse=True)[:5]:
    if evolucao[t]['titulo'][-1] > 0.5:
        delta = evolucao[t]['titulo'][-1] - evolucao[t]['titulo'][0]
        seta = '↑' if delta > 0 else ('↓' if delta < 0 else '→')
        print(f'  {t:<20s} {evolucao[t]["titulo"][-1]:5.1f}%  ({seta} {delta:+.1f}pp desde R1)')

print(f'\nRisco de REBAIXAMENTO após rodada {rodada_max}:')
for t in sorted(times_2026, key=lambda t: evolucao[t]['z4'][-1], reverse=True)[:5]:
    if evolucao[t]['z4'][-1] > 5:
        delta = evolucao[t]['z4'][-1] - evolucao[t]['z4'][0]
        seta = '↑' if delta > 0 else ('↓' if delta < 0 else '→')
        print(f'  {t:<20s} {evolucao[t]["z4"][-1]:5.1f}%  ({seta} {delta:+.1f}pp desde R1)')

print(f'\n{len(rodadas_simuladas)} rodadas × {N_SIMULACOES:,} simulações = {len(rodadas_simuladas) * N_SIMULACOES:,} cenários analisados')
print(f'\n6 gráficos exportados para charts/')

EVOLUÇÃO DAS PREVISÕES — Brasileirão 2026
Rodadas analisadas: 1 a 6

Chance de TÍTULO após rodada 6:
  Palmeiras             53.0%  (↑ +10.2pp desde R1)
  Flamengo              25.1%  (↑ +8.9pp desde R1)
  Fluminense             6.3%  (↑ +4.8pp desde R1)
  Botafogo               6.0%  (↓ -19.7pp desde R1)
  Mirassol               5.4%  (↓ -2.3pp desde R1)

Risco de REBAIXAMENTO após rodada 6:
  Remo                  68.6%  (↑ +42.3pp desde R1)
  Santos                62.2%  (↑ +18.5pp desde R1)
  Coritiba              52.4%  (↓ -38.3pp desde R1)
  Vasco                 46.1%  (↑ +13.8pp desde R1)
  Vitória               41.2%  (↓ -13.9pp desde R1)

6 rodadas × 5,000 simulações = 30,000 cenários analisados

6 gráficos exportados para charts/
